# Natural Language to SPARQL with Ollama

This notebook demonstrates how to translate natural language questions into **SPARQL 1.1** queries
using a locally-hosted LLM via **Ollama**. SPARQL is the W3C standard query language for RDF
knowledge graphs, used with triple stores like Apache Jena Fuseki, Blazegraph, and GraphDB.

The pipeline uses:
- **OWL/Turtle TBox injection** to ground the model in the ontology
- **Pydantic structured output** with Ollama's grammar-constrained decoding
- **RDFLib validation** to catch syntax errors before execution
- A **self-correction retry loop** for robust query generation

## Prerequisites

1. Install and run [Ollama](https://ollama.com/)
2. Pull a suitable model: `ollama pull qwen2.5:1.5b`
3. Install Python dependencies: `pip install ollama pydantic rdflib requests`

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install ollama pydantic rdflib requests

## Step 0: Connect to Ollama Endpoint
If you are using an Ollama endpoint, either Ollama Cloud or a dedicated server, you need to set the endpoint's base URL and your APU key.
1. Save your API key in a file outside of your project directory, that you can load into the notebook. **Never include API keys directly into your notebooks or program code!**
2. Define the variables `OLLAMA_BASE_URL` and `OLLAMA_API_KEY` to your configuration.

The following code tests your Ollama connection by listing your installed models.

In [ ]:
import os
OLLAMA_BASE_URL="https://gpu-01.insight.gsu.edu:11443"
OLLAMA_API_KEY=open(os.path.expanduser("~/.secrets/insight_ollama_msa8700_student.txt"), 'r').read().strip()
print(f"Length of API Key: {len(OLLAMA_API_KEY)}")

# from ollama import chat
from ollama import Client

client = Client(
    host=OLLAMA_BASE_URL,
    headers={'Authorization': f'Bearer {OLLAMA_API_KEY}'}
)

import pandas as pd
import json
response = client.list()
print(f"Number of models: {len(response.models):,}")
# models = pd.DataFrame.from_records([m.model_dump() for m in response.models])
# for col in ['parent_model', 'format', 'family', 'families', 'parameter_size', 'quantization_level']:
#     models[col] = models.apply(lambda m: m.details[col], axis=1)

# display(models[['model', 'size', 'family', 'parameter_size', 'quantization_level']])

## Step 1: Define the OWL/Turtle TBox (Ontology)

The TBox defines the vocabulary of the knowledge graph — classes, object properties, and datatype
properties. This is injected into the prompt so the model can **only** reference classes and
properties that actually exist in the graph. Missing schema elements lead to hallucinated
class/property names.

In [ ]:
TBOX_TURTLE = """
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .
@prefix ex:   <http://example.org/ontology#> .

### Classes
ex:Person        a owl:Class .
ex:Researcher    a owl:Class ; rdfs:subClassOf ex:Person .
ex:Organization  a owl:Class .
ex:Publication   a owl:Class .

### Object Properties
ex:worksAt        a owl:ObjectProperty ;
                  rdfs:domain ex:Person ;
                  rdfs:range  ex:Organization .
ex:authored       a owl:ObjectProperty ;
                  rdfs:domain ex:Researcher ;
                  rdfs:range  ex:Publication .
ex:affiliatedWith a owl:ObjectProperty ;
                  rdfs:domain ex:Publication ;
                  rdfs:range  ex:Organization .

### Datatype Properties
ex:name      a owl:DatatypeProperty ; rdfs:range xsd:string .
ex:birthYear a owl:DatatypeProperty ; rdfs:range xsd:integer .
ex:title     a owl:DatatypeProperty ; rdfs:range xsd:string .
ex:year      a owl:DatatypeProperty ; rdfs:range xsd:integer .
ex:industry  a owl:DatatypeProperty ; rdfs:range xsd:string .
ex:founded   a owl:DatatypeProperty ; rdfs:range xsd:integer .
"""

print("TBox loaded. Classes: Person, Researcher, Organization, Publication")

## Step 2: Define the Structured Output Schema

The Pydantic model captures not just the query, but metadata that is useful for debugging
and observability:
- `query_type` — SELECT, ASK, CONSTRUCT, or DESCRIBE
- `classes_used` — which TBox classes the query references
- `explanation` — what the model thinks the query does (chain-of-thought)

In [ ]:
from pydantic import BaseModel


class SPARQLResult(BaseModel):
    query: str               # Full SPARQL query including PREFIX declarations
    query_type: str          # "SELECT" | "ASK" | "CONSTRUCT" | "DESCRIBE"
    classes_used: list[str]  # TBox classes referenced, e.g. ["ex:Researcher"]
    explanation: str         # One-sentence description of what the query does

## Step 3: Build the System Prompt

The system prompt includes:
1. The model's role and behavioral constraints
2. The full TBox ontology
3. Two few-shot examples showing the expected NL-to-SPARQL pattern

The few-shot examples are critical for anchoring SPARQL-specific syntax like `PREFIX`,
`SELECT DISTINCT`, `OPTIONAL`, and `FILTER`.

In [ ]:
SPARQL_SYSTEM_PROMPT = f"""You are an expert SPARQL 1.1 query generator for RDF knowledge graphs.
Given an OWL/Turtle ontology TBox and a natural language question, generate a valid SPARQL query.

Rules:
- Use ONLY the classes and properties defined in the TBox
- Always declare PREFIX statements at the top
- Use OPTIONAL {{{{ }}}} for properties that may not exist on all instances
- Use FILTER for conditions on literals
- Use SELECT DISTINCT unless aggregation is required
- For aggregations, use GROUP BY with the appropriate aggregate function

Ontology TBox (Turtle format):
{TBOX_TURTLE}

Examples:
Q: Who are all the researchers?
SPARQL:
PREFIX ex: <http://example.org/ontology#>
SELECT DISTINCT ?name WHERE {{{{
  ?r a ex:Researcher ; ex:name ?name .
}}}}

Q: Which organizations are in the healthcare industry?
SPARQL:
PREFIX ex: <http://example.org/ontology#>
SELECT DISTINCT ?orgName WHERE {{{{
  ?o a ex:Organization ; ex:industry "healthcare" ; ex:name ?orgName .
}}}}
"""

## Step 4: SPARQL Validation with RDFLib

RDFLib's `prepareQuery()` function parses and compiles a SPARQL query against the SPARQL 1.1
grammar **without executing it**. This is pure Python validation — no triple store required.

In [ ]:
from rdflib.plugins.sparql import prepareQuery


def validate_sparql(query: str) -> tuple[bool, str]:
    """Validate SPARQL syntax using RDFLib."""
    try:
        prepareQuery(query)
        return True, "Valid"
    except Exception as e:
        return False, str(e)

In [ ]:
# Quick test: valid SPARQL
valid_query = """PREFIX ex: <http://example.org/ontology#>
SELECT DISTINCT ?name WHERE {
  ?r a ex:Researcher ; ex:name ?name .
}"""
print("Valid:", validate_sparql(valid_query))

# Quick test: invalid SPARQL (missing closing brace)
invalid_query = """PREFIX ex: <http://example.org/ontology#>
SELECT ?name WHERE {
  ?r a ex:Researcher ; ex:name ?name .
"""
print("Invalid:", validate_sparql(invalid_query))

## Step 5: Generation with Self-Correction Loop

The generation function sends the question to Ollama with the structured output schema.
If validation fails, the error is fed back to the model for correction.

In [ ]:
MODEL = "qwen2.5:1.5b"


def nl_to_sparql(question: str, max_retries: int = 3) -> SPARQLResult:
    """Translate a natural language question to a validated SPARQL query."""
    error_context = ""

    for attempt in range(1, max_retries + 1):
        user_content = f"Question: {question}"
        if error_context:
            user_content += (
                f"\n\nYour previous attempt produced an invalid SPARQL query.\n"
                f"Error: {error_context}\n"
                f"Please correct the query."
            )

        response = client.chat(
            model=MODEL,
            messages=[
                {"role": "system", "content": SPARQL_SYSTEM_PROMPT},
                {"role": "user", "content": user_content},
            ],
            format=SPARQLResult.model_json_schema(),
            options={"temperature": 0.0},
        )

        result = SPARQLResult.model_validate_json(response.message.content)
        is_valid, error = validate_sparql(result.query)

        status = "Valid" if is_valid else f"Error: {error}"
        print(f"  [Attempt {attempt}] {status}")

        if is_valid:
            return result
        error_context = error

    raise ValueError(f"Failed to generate valid SPARQL after {max_retries} attempts.")

## Step 6: Run Examples

Let's test the pipeline with several questions against our research ontology.

In [ ]:
questions = [
    "List all researchers and the organizations they work at.",
    "How many publications were authored by researchers in the AI industry?",
    "Find all publications affiliated with organizations founded before 2000.",
    "Who are the researchers born after 1985?",
    "Which organizations have researchers who have authored publications?",
]

In [ ]:
for q in questions:
    print(f"\n{'=' * 70}")
    print(f"Question: {q}")
    print("-" * 70)

    result = nl_to_sparql(q)

    print(f"\nQuery Type : {result.query_type}")
    print(f"Classes    : {result.classes_used}")
    print(f"Explanation: {result.explanation}")
    print(f"\nGenerated SPARQL:\n{result.query}")

## Step 7: Inspect Structured Output Detail

Let's look at the full JSON structure returned for a single query.

In [ ]:
import json

result = nl_to_sparql("How many publications has each researcher authored?")
print(json.dumps(result.model_dump(), indent=2))

## Step 8 (Optional): Execute Against Apache Jena Fuseki

If you have a Fuseki SPARQL endpoint running, you can execute the generated queries directly.
Fuseki can be started with:

```bash
# Download and start Fuseki with an in-memory dataset
fuseki-server --mem /ds
```

The SPARQL endpoint will be available at `http://localhost:3030/ds/sparql`.

In [ ]:
import requests


def run_sparql(
    query: str, endpoint: str = "http://localhost:3030/ds/sparql"
) -> list[dict]:
    """Execute a SPARQL query against a Fuseki endpoint."""
    resp = requests.get(
        endpoint,
        params={"query": query},
        headers={"Accept": "application/sparql-results+json"},
        timeout=30,
    )
    resp.raise_for_status()
    bindings = resp.json().get("results", {}).get("bindings", [])
    return [{k: v["value"] for k, v in row.items()} for row in bindings]


# Uncomment to execute against a live Fuseki endpoint:
# result = nl_to_sparql("List all researchers and their publications.")
# rows = run_sparql(result.query)
# for row in rows:
#     print(row)

## Key Takeaways

1. **TBox injection** is the SPARQL equivalent of DDL schema injection for SQL — it grounds the model in the ontology vocabulary.
2. **RDFLib's `prepareQuery()`** validates SPARQL syntax without needing a running triple store.
3. **Structured output metadata** (`query_type`, `classes_used`) provides observability for production pipelines.
4. **The retry loop** with error feedback enables self-correction of syntax issues.
5. For large ontologies, consider a **RAG step over the TBox** — embed class/property definitions and retrieve only the top-K relevant chunks per query.